In [ ]:
from pathlib import Path
import csv, subprocess, sys
import torch
assert torch.cuda.is_available(), 'Stage A requires GPU'
repo=Path('/tmp/PCC')
subprocess.run(['git','clone','--quiet','https://github.com/changxinjiresearch/PCC.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--quiet','66269681e4417dccabc68ecaa792d76e19aa5856'],check=True)
protocol=repo/'outputs/pcc_115_holdout_protocol_lock_2026'
stage_manifest=protocol/'09_EXECUTION_PLAN/LOCKED_115_STAGE_A_P0_SHARD_MANIFEST.csv'
with stage_manifest.open(newline='') as h: rows=list(csv.DictReader(h))
assert len(rows)==115 and len({r['patient_id'] for r in rows})==115 and not any(any(t in (k+' '+v).lower() for t in ('future','target','later','ground_truth','outcome','progression_truth')) for r in rows for k,v in r.items())
out=Path('/kaggle/working/pcc_115_holdout_stage_a_p0_freeze_2026')
sys.path.insert(0,str(repo))
from experiments.run_115_stage_a_p0 import execute_stage_a_all_shards
execute_stage_a_all_shards(stage_manifest,protocol/'03_PREDICTOR_LOCK/LOCKED_115_CHECKPOINT_MANIFEST.csv',out)
subprocess.run([sys.executable,'-m','experiments.finalize_stage_a_p0_kernel_output'],cwd=repo,check=True)
